In [1]:
#!pip install dash==2.6.0
#!pip install dash_bootstrap_components
#!pip install jupyter_dash


2023-07-07 19:00:04.904899: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-07 19:00:04.913580: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-07-07 19:00:04.913617: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
import datetime
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import re
import tensorflow as tf
import json
import plotly.graph_objects as go

import plotly.express as px
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
from jupyter_dash import JupyterDash

import sys
sys.path.append('./rtaUtils')
from rtaUtils import data_loading, common, sort_vectors, paths, experiment, data_preparation

lookf_actual=10
config = {
  'toImageButtonOptions': {
    'format': 'png', # one of png, svg, jpeg, webp
    'filename': 'newplot',
    'height': 1000,
    'width': 1000,
    'scale':1 # Multiply title/legend/axis/canvas sizes by this factor
  }
}

airport_list = ('LEBL','LPPT','LFPO','LFPG','EGLL','LPPR','EDDF','EBBR','EHAM','LEBB','LIRF',
                'EDDM','EGKK','LEVC','LSGG','LECO','LIMC','EIDW','LFML','LEZL','LOWW','LIPZ',
                'LIPE','LFLL','EDDL','EDDB','LTFM','LFMN','LFRS','LROP','EDDP','LGAV','EDDH',
                'LHBP','EKCH','EGCC','ELLX','LKPR','LIRN','LBSF','EPWA')

numeric_feat   = ['latitude', 'longitude', 'altitude'] 
categoric_feat = [] #'operator'      
objective      = ['latitude', 'longitude', 'altitude']

feat_dict = dict(
    numeric=numeric_feat,
    categoric=categoric_feat,
    objective=objective
)

In [3]:
sampling = 15
months = '*'

In [4]:
models = [x.stem for x in paths.models_path.glob('*/')]

experimento = None
month_data = None
tray_data = None
prediction = None
windows = None
prediction_windows = None
true_windows = None

current_month = '202201'

## Inicialización

In [5]:
# Estilos de los ejemplos de Dash
external_stylesheets = [dbc.themes.BOOTSTRAP,'https://codepen.io/chriddyp/pen/bWLwgP.css']

app = JupyterDash(__name__, external_stylesheets=external_stylesheets)

## Estructura

In [6]:
############### Model ###############

modelSelector = dcc.Dropdown(
    id = 'modelSelector',
    options={x:x.upper().replace('_', ' ') for x in models},
    value=models[0],
)

modelVersionSelector = dcc.Dropdown(
    id = 'modelVersionSelector',
)

############### Trajectories ###############

dateSelector = dcc.DatePickerSingle(
    id = 'dateSelector',
    placeholder = 'Select one...',
    date='2022-01-01',
#     initial_visible_month=data.flightDate.min(),
    min_date_allowed = '2022-01-01',
    max_date_allowed = '2022-09-30',
    display_format='DD/MM/YYYY',
#     disabled_days=dates
)

airportSelector = dcc.Dropdown(
    id = 'airportSelector',
    options = [dict(label=x, value=x) for x in sorted(airport_list)],
    placeholder='Select one...',
    multi=True,
)

trajectorySelector = dcc.Dropdown(
    id = 'trajectorySelector',
    placeholder='Select one...',
    multi=False,
)

windowSelector = dcc.Input(
    id = 'windowSelector',
    type = 'number',
    min=0, max=100,
    value=0,
    #tooltip={'placement': 'top', 'always_visible': False},
    step=1
)

In [7]:
app.layout = html.Div(style=dict(display='flex'), children=[
    html.Div(style=dict(width='20%', padding=5,), children=[
        html.Fieldset(style=dict(width='99%', borderWidth=1, borderStyle='solid', 
                                 borderRadius=5, padding=5, margin=2), children=[
            html.Legend('Data', style=dict(fontWeight='bold')),
            dbc.Row(style=dict(paddingTop=5, paddingBottom=5), children=[
                dbc.Col(dbc.Label('Months'), width=4),
                dbc.Col(months, width=7),
                dbc.Col(dbc.Label('Sampling'), width=4),
                dbc.Col(sampling, width=7),
            ]),
        ]),
        
        html.Fieldset(style=dict(width='99%', borderWidth=1, borderStyle='solid', 
                                 borderRadius=5, padding=5, margin=2), children=[
            html.Legend('Model', style=dict(fontWeight='bold')),
            dbc.Row(style=dict(paddingTop=5, paddingBottom=5), children=[
                dbc.Col([dbc.Label('Model'), modelSelector], width=12),
            ]),
            dbc.Row(style=dict(paddingTop=5, paddingBottom=5), children=[
                dbc.Col([dbc.Label('Version'), modelVersionSelector], width=12),
            ]),
            html.Div(id='placeholder', style={'display':'none'})
        ]),
        
        html.Fieldset(style=dict(width='99%', borderWidth=1, borderStyle='solid', 
                                 borderRadius=5, padding=5, margin=2), children=[
            html.Legend('Trajectory', style=dict(fontWeight='bold')),
            dbc.Row(style=dict(paddingTop=5, paddingBottom=5), children=[
                dbc.Col([dbc.Label('Date'), dateSelector], width=6),
                dbc.Col([dbc.Label('Airport'), airportSelector], width=6),
            ]),
            dbc.Row(style=dict(paddingTop=5, paddingBottom=5), children=[
                dbc.Col([dbc.Label('Trajectory'), trajectorySelector], width=12),
            ]),
        ]),
        
        html.Fieldset(style=dict(width='99%', borderWidth=1, borderStyle='solid', 
                                 borderRadius=5, padding=5, margin=2), children=[
            html.Legend('Experiment', style=dict(fontWeight='bold')),
            dcc.RadioItems(
                id='experimentTypeSelector',
                style=dict(padding=5),
                options=dict(full=' Full trajectory', window=' Individual window'),
                value='full',
            ),
            html.Div(id='sliderDiv', children=[windowSelector]),
            html.Button('Run experiment', id='runButton', 
                        style=dict(margin='auto', marginTop=20, display='flex', alignItems='center')),
        ]),
        
        html.Fieldset(style=dict(width='99%', borderWidth=1, borderStyle='solid', 
                                 borderRadius=5, padding=5, margin=2), children=[
            html.Legend('Map type', style=dict(fontWeight='bold')),
            dcc.RadioItems(
                id='mapTypeSelector',
                style=dict(padding=5),
                options=['open-street-map',
                         'carto-positron', 
                         'carto-darkmatter', 
                         'stamen-terrain', 
                         'stamen-toner'],
                value='open-street-map',
            ),
        ]),
    ]),
    
    html.Div(id='mapDiv', style=dict(width='75%', margin=7, borderWidth=1, borderStyle='solid', borderRadius=5), children=[
        dcc.Graph(id='mapGraph', style=dict(width='100%',height='100%'),)
    ])
])

In [8]:
@app.callback(
    [Output(component_id='modelVersionSelector', component_property='options'),
     Output(component_id='modelVersionSelector', component_property='value'),
     Output(component_id='dateSelector', component_property='date')],
    [Input(component_id='modelSelector', component_property='value')],
    [State(component_id='dateSelector', component_property='date')] 
)
def load_model(model_name,current_date):
    global model
    global experimento
    model_path = './models/'
    
    if model_name:
        with open(model_path + model_name + "/experiment_config.json", 'r') as input_file:
            data = json.load(input_file)
        conf = model_name.split('_')
        experimento = experiment.ExperimentTrajectory(
            model_type = data['model_type'],
            lookback=data['lookback'],
            lookforward=data['lookforward'],
            sampling=data['sampling'],
            model_config=dict(n_units=data['num_units'], 
                              act_function = data['activation_function'],
                              batch_size   = data['batch_size']),
            months=data['months'], 
            airport=data['airport'],
            features=data['features']
        )
        experimento.load_model('best')
        
        model_versions = {x.stem:x.stem.replace('_', ' ') for x in (paths.models_path / model_name).glob('*.h5')}
    else: 
        model_versions={'best':'Best'}
    
    return (model_versions,'best',current_date)


@app.callback(
    [Output(component_id='placeholder', component_property='children')],
    [Input(component_id='modelVersionSelector', component_property='value')],
    []
)
def load_model_version(model_version):
    global experimento
    
    if model_version:
        experimento.load_model(model_version)
    return ([],)


@app.callback(
    [Output(component_id='trajectorySelector', component_property='options'),
     Output(component_id='trajectorySelector', component_property='value')],
    [Input(component_id='dateSelector', component_property='date'),
     Input(component_id='airportSelector', component_property='value')],
    [State(component_id='trajectorySelector', component_property='value')]
)
def filter_trajectories(fecha, origen,current_trajectory):
    global month_data
    global current_month
    if fecha:
        m = fecha[:8].replace('-','')

        #if month_data is None or current_month is None or m != current_month:
        month_data = data_loading.load_final_data(m, 'test', sampling=sampling)
        current_month = m
        df = month_data[pd.to_datetime(month_data.timestamp, unit='s').dt.date.astype(str) == fecha].copy()
        if origen:
            df = df[df.aerodromeOfDeparture.isin(origen)]

        labels = df[['fpId','aerodromeOfDeparture']].drop_duplicates().sort_values(['aerodromeOfDeparture'])
        labels = [{'label': f'{x[1].aerodromeOfDeparture} {x[1].fpId}', 'value': x[1].fpId} 
                   for x in labels.iterrows()]
        
        return [labels,current_trajectory]
    else:
        return [{},None]

    
@app.callback(
    [Output(component_id='windowSelector', component_property='max')],
    [Input(component_id='trajectorySelector', component_property='value')],
    []
)
def prepare_experiment(trajectory):
    global month_data
    global experimento
    global tray_data
    global prediction
    global windows
    global true_windows
    
    if experimento is None or month_data is None or trajectory is None:
        return [0]
    
    # Full predictions
    tray_data = month_data[month_data.fpId == trajectory].copy()
    prediction = experimento.predict_trajectory(tray_data.copy())
    #me quedo con las predicciones que debo pintar
    auxDF = pd.DataFrame()
    for i in range(0, len(prediction)-experimento.lookforward,experimento.lookforward*experimento.lookforward):
        auxDF = auxDF.append(prediction.iloc[list(range(i,i+experimento.lookforward))])
        
    prediction = auxDF.copy()
    prediction['point'] = 'predicted'
    
    # Predictions for individual windows
    windows = [tray_data.iloc[i:i+experimento.lookback,:].copy()
               for i in range(tray_data.shape[0]-experimento.lookback-experimento.lookforward-experimento.shift)]
    
    true_windows = [tray_data.iloc[i-1:i+lookf_actual-1,:].copy()
                    for i in range(experimento.lookback+experimento.shift+1, tray_data.shape[0])]

    if tray_data is not None:
        max_value = len(windows)-1

    return [max_value]


@app.callback(
    [Output(component_id='sliderDiv', component_property='style')],
    [Input(component_id='experimentTypeSelector', component_property='value')],
    [])
def change_experiment_type(exp_type):
    if exp_type == 'full':
        return ({'display':'none',},)
    else:
        return ({'display':'block',},)

In [9]:
@app.callback(
    [Output(component_id='mapGraph', component_property='figure')],
    [Input(component_id='runButton', component_property='n_clicks'),
     Input(component_id='windowSelector', component_property='value'),
     Input(component_id='mapTypeSelector', component_property='value'),
     Input(component_id='mapGraph', component_property='relayoutData')
    ],
    [State(component_id='experimentTypeSelector', component_property='value'),
     State(component_id='mapGraph', component_property='figure')]
)
def run_experiment(clicks, selected_window, map_type, relData, exp_type, current_fig):
    global experimento
    global tray_data
    global prediction
    global windows
    global true_windows
    
    
    if selected_window is None:
        return (current_fig,)
    
    elif clicks and experimento:
            
    
        current_zoom = current_fig['layout']['mapbox']['zoom']
        current_center = current_fig['layout']['mapbox']['center']
        df_viz = tray_data[experimento.objective_feat].copy()
        df_viz['point'] = 'real'
               
        if exp_type == 'window':
            predictions = experimento.predict_trajectory_acumulado(windows[selected_window].copy(),lookf_actual)
            window = windows[selected_window][experimento.objective_feat].copy()
            window['point'] = 'window'
            predictions['point'] = 'predicted'
            true = true_windows[selected_window][experimento.objective_feat].copy()
            true['point'] = 'true'
            
            true_array = true[['latitude','longitude','altitude']].to_numpy()
            true_array = true_array.reshape(-1,len(experimento.objective_feat))

            true_array = true_array.reshape((-1, lookf_actual, len(experimento.objective_feat)))
            predictions_array = predictions[['latitude','longitude','altitude']].to_numpy()
            predictions_array = predictions_array.reshape(-1,len(experimento.objective_feat))
            predictions_array = predictions_array.reshape((-1, experimento.lookforward, len(experimento.objective_feat)))
            
            df_viz = pd.concat([df_viz.iloc[:selected_window], 
                                window,
                                true,
                                predictions,
                                df_viz.iloc[selected_window+experimento.lookback+experimento.lookforward+experimento.shift:]], axis=0)
        elif exp_type == 'full':
            predictions = prediction
            df_viz = pd.concat([df_viz, predictions], axis=0)
        
        
        fig = px.scatter_mapbox(
            df_viz, 'latitude', 'longitude', height=850, zoom = current_zoom, center = current_center, #zoom=4,
            mapbox_style=map_type, #opacity = 1,
            title='Map',
            color ='point', hover_data = ['altitude']
        )
        if predictions is not None and exp_type == 'window':
            fig.add_trace(go.Scattermapbox(
              mode = "markers+lines", lon = predictions.longitude, lat = predictions.latitude,
               showlegend=False, marker = {'size': 3, 'color': '#ab63fa'},
            ))
        elif predictions is not None and exp_type == 'full':
            fig.add_trace(go.Scattermapbox(
              mode = "markers+lines", lon = predictions.longitude, lat = predictions.latitude,
               showlegend=False, marker = {'size': 3, 'color': '#ef553b'},
            ))
        # px.scatter_geo()
        points_size = current_zoom*1.5
        fig.update_traces(marker={'size':points_size})
        return (fig,)
    else:
        return (px.scatter_mapbox(lat=[43.0],lon=[4.0], zoom = 4, mapbox_style='open-street-map', title='Map'),)

## Callbacks

In [10]:
app.run_server(debug = True)
# mode='inline'

Dash app running on http://127.0.0.1:8050/


2023-07-07 19:00:07.870086: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2023-07-07 19:00:07.870170: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2023-07-07 19:00:07.870214: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (tfg-6538): /proc/driver/nvidia/version does not exist
2023-07-07 19:00:07.870868: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


           fpId  icao24 callsign   latitude  longitude  speed  vspeed  ground  \
802  AT05429271  346499  IBE31LR  51.285099     6.7632  143.0  2752.0   False   
803  AT05429271  346499  IBE31LR  51.278999     6.7505  135.0  3201.0   False   
804  AT05429271  346499  IBE31LR  51.272099     6.7360  143.0  2496.0   False   
805  AT05429271  346499  IBE31LR  51.266602     6.7246  154.0  2303.0   False   
806  AT05429271  346499  IBE31LR  51.259201     6.7087  178.0  2175.0   False   
807  AT05429271  346499  IBE31LR  51.251598     6.6924  197.0  3904.0   False   
808  AT05429271  346499  IBE31LR  51.243301     6.6738  216.0  2817.0   False   
809  AT05429271  346499  IBE31LR  51.238499     6.6491  238.0  2817.0   False   
810  AT05429271  346499  IBE31LR  51.241501     6.6236  259.0  3264.0   False   
811  AT05429271  346499  IBE31LR  51.254299     6.5994  269.0  4799.0   False   
812  AT05429271  346499  IBE31LR  51.269199     6.5812  267.0  4799.0   False   
813  AT05429271  346499  IBE

           fpId  icao24 callsign   latitude  longitude  speed  vspeed  ground  \
803  AT05429271  346499  IBE31LR  51.278999     6.7505  135.0  3201.0   False   
804  AT05429271  346499  IBE31LR  51.272099     6.7360  143.0  2496.0   False   
805  AT05429271  346499  IBE31LR  51.266602     6.7246  154.0  2303.0   False   
806  AT05429271  346499  IBE31LR  51.259201     6.7087  178.0  2175.0   False   
807  AT05429271  346499  IBE31LR  51.251598     6.6924  197.0  3904.0   False   
808  AT05429271  346499  IBE31LR  51.243301     6.6738  216.0  2817.0   False   
809  AT05429271  346499  IBE31LR  51.238499     6.6491  238.0  2817.0   False   
810  AT05429271  346499  IBE31LR  51.241501     6.6236  259.0  3264.0   False   
811  AT05429271  346499  IBE31LR  51.254299     6.5994  269.0  4799.0   False   
812  AT05429271  346499  IBE31LR  51.269199     6.5812  267.0  4799.0   False   
813  AT05429271  346499  IBE31LR  51.284100     6.5644  274.0  3776.0   False   
814  AT05429271  346499  IBE

           fpId  icao24 callsign   latitude  longitude  speed  vspeed  ground  \
803  AT05429271  346499  IBE31LR  51.278999     6.7505  135.0  3201.0   False   
804  AT05429271  346499  IBE31LR  51.272099     6.7360  143.0  2496.0   False   
805  AT05429271  346499  IBE31LR  51.266602     6.7246  154.0  2303.0   False   
806  AT05429271  346499  IBE31LR  51.259201     6.7087  178.0  2175.0   False   
807  AT05429271  346499  IBE31LR  51.251598     6.6924  197.0  3904.0   False   
808  AT05429271  346499  IBE31LR  51.243301     6.6738  216.0  2817.0   False   
809  AT05429271  346499  IBE31LR  51.238499     6.6491  238.0  2817.0   False   
810  AT05429271  346499  IBE31LR  51.241501     6.6236  259.0  3264.0   False   
811  AT05429271  346499  IBE31LR  51.254299     6.5994  269.0  4799.0   False   
812  AT05429271  346499  IBE31LR  51.269199     6.5812  267.0  4799.0   False   
813  AT05429271  346499  IBE31LR  51.284100     6.5644  274.0  3776.0   False   
814  AT05429271  346499  IBE

...

            fpId  icao24 callsign   latitude  longitude  speed  vspeed  \
1593  AT05429397  3415CE  IBE3237  40.620300    -1.6190  433.0 -2303.0   
1594  AT05429397  3415CE  IBE3237  40.612598    -1.6503  430.0 -2817.0   
1595  AT05429397  3415CE  IBE3237  40.600201    -1.7018  423.0 -2303.0   
1596  AT05429397  3415CE  IBE3237  40.593700    -1.7283  417.0 -2431.0   
1597  AT05429397  3415CE  IBE3237  40.585899    -1.7606  413.0 -2496.0   
1598  AT05429397  3415CE  IBE3237  40.577000    -1.7973  410.0 -2880.0   
1599  AT05429397  3415CE  IBE3237  40.568100    -1.8339  407.0 -2689.0   
1600  AT05429397  3415CE  IBE3237  40.559601    -1.8688  403.0 -2368.0   
1601  AT05429397  3415CE  IBE3237  40.550400    -1.9064  394.0 -2175.0   
1602  AT05429397  3415CE  IBE3237  40.542301    -1.9398  390.0 -2880.0   
1603  AT05429397  3415CE  IBE3237  40.531200    -1.9850  388.0 -2689.0   
1604  AT05429397  3415CE  IBE3237  40.526299    -2.0056  387.0 -1984.0   
1605  AT05429397  3415CE  IBE3237  40.

            fpId  icao24 callsign   latitude  longitude  speed  vspeed  \
1594  AT05429397  3415CE  IBE3237  40.612598    -1.6503  430.0 -2817.0   
1595  AT05429397  3415CE  IBE3237  40.600201    -1.7018  423.0 -2303.0   
1596  AT05429397  3415CE  IBE3237  40.593700    -1.7283  417.0 -2431.0   
1597  AT05429397  3415CE  IBE3237  40.585899    -1.7606  413.0 -2496.0   
1598  AT05429397  3415CE  IBE3237  40.577000    -1.7973  410.0 -2880.0   
1599  AT05429397  3415CE  IBE3237  40.568100    -1.8339  407.0 -2689.0   
1600  AT05429397  3415CE  IBE3237  40.559601    -1.8688  403.0 -2368.0   
1601  AT05429397  3415CE  IBE3237  40.550400    -1.9064  394.0 -2175.0   
1602  AT05429397  3415CE  IBE3237  40.542301    -1.9398  390.0 -2880.0   
1603  AT05429397  3415CE  IBE3237  40.531200    -1.9850  388.0 -2689.0   
1604  AT05429397  3415CE  IBE3237  40.526299    -2.0056  387.0 -1984.0   
1605  AT05429397  3415CE  IBE3237  40.517799    -2.0423  386.0  -896.0   
1606  AT05429397  3415CE  IBE3237  40.

            fpId  icao24 callsign   latitude  longitude  speed  vspeed  \
1594  AT05429397  3415CE  IBE3237  40.612598    -1.6503  430.0 -2817.0   
1595  AT05429397  3415CE  IBE3237  40.600201    -1.7018  423.0 -2303.0   
1596  AT05429397  3415CE  IBE3237  40.593700    -1.7283  417.0 -2431.0   
1597  AT05429397  3415CE  IBE3237  40.585899    -1.7606  413.0 -2496.0   
1598  AT05429397  3415CE  IBE3237  40.577000    -1.7973  410.0 -2880.0   
1599  AT05429397  3415CE  IBE3237  40.568100    -1.8339  407.0 -2689.0   
1600  AT05429397  3415CE  IBE3237  40.559601    -1.8688  403.0 -2368.0   
1601  AT05429397  3415CE  IBE3237  40.550400    -1.9064  394.0 -2175.0   
1602  AT05429397  3415CE  IBE3237  40.542301    -1.9398  390.0 -2880.0   
1603  AT05429397  3415CE  IBE3237  40.531200    -1.9850  388.0 -2689.0   
1604  AT05429397  3415CE  IBE3237  40.526299    -2.0056  387.0 -1984.0   
1605  AT05429397  3415CE  IBE3237  40.517799    -2.0423  386.0  -896.0   
1606  AT05429397  3415CE  IBE3237  40.

            fpId  icao24 callsign   latitude  longitude  speed  vspeed  \
1594  AT05429397  3415CE  IBE3237  40.612598    -1.6503  430.0 -2817.0   
1595  AT05429397  3415CE  IBE3237  40.600201    -1.7018  423.0 -2303.0   
1596  AT05429397  3415CE  IBE3237  40.593700    -1.7283  417.0 -2431.0   
1597  AT05429397  3415CE  IBE3237  40.585899    -1.7606  413.0 -2496.0   
1598  AT05429397  3415CE  IBE3237  40.577000    -1.7973  410.0 -2880.0   
1599  AT05429397  3415CE  IBE3237  40.568100    -1.8339  407.0 -2689.0   
1600  AT05429397  3415CE  IBE3237  40.559601    -1.8688  403.0 -2368.0   
1601  AT05429397  3415CE  IBE3237  40.550400    -1.9064  394.0 -2175.0   
1602  AT05429397  3415CE  IBE3237  40.542301    -1.9398  390.0 -2880.0   
1603  AT05429397  3415CE  IBE3237  40.531200    -1.9850  388.0 -2689.0   
1604  AT05429397  3415CE  IBE3237  40.526299    -2.0056  387.0 -1984.0   
1605  AT05429397  3415CE  IBE3237  40.517799    -2.0423  386.0  -896.0   
1606  AT05429397  3415CE  IBE3237  40.

            fpId  icao24 callsign   latitude  longitude  speed  vspeed  \
1594  AT05429397  3415CE  IBE3237  40.612598    -1.6503  430.0 -2817.0   
1595  AT05429397  3415CE  IBE3237  40.600201    -1.7018  423.0 -2303.0   
1596  AT05429397  3415CE  IBE3237  40.593700    -1.7283  417.0 -2431.0   
1597  AT05429397  3415CE  IBE3237  40.585899    -1.7606  413.0 -2496.0   
1598  AT05429397  3415CE  IBE3237  40.577000    -1.7973  410.0 -2880.0   
1599  AT05429397  3415CE  IBE3237  40.568100    -1.8339  407.0 -2689.0   
1600  AT05429397  3415CE  IBE3237  40.559601    -1.8688  403.0 -2368.0   
1601  AT05429397  3415CE  IBE3237  40.550400    -1.9064  394.0 -2175.0   
1602  AT05429397  3415CE  IBE3237  40.542301    -1.9398  390.0 -2880.0   
1603  AT05429397  3415CE  IBE3237  40.531200    -1.9850  388.0 -2689.0   
1604  AT05429397  3415CE  IBE3237  40.526299    -2.0056  387.0 -1984.0   
1605  AT05429397  3415CE  IBE3237  40.517799    -2.0423  386.0  -896.0   
1606  AT05429397  3415CE  IBE3237  40.